# Seam Harmonizer Model Surgery Colab

Готовый notebook для запуска `model_surgery` на Colab GPU.

Сценарий:
- clone репозитория из git;
- копирование `synthetic_triplets_500` с Google Drive в локальный диск или RAM;
- копирование training checkpoints / runs локально, чтобы не читать их с Drive во время eval;
- сборка runtime-конфига для `model_surgery`;
- запуск `python -m model_surgery.run`;
- синк результатов обратно на Drive.

In [ ]:
# 0. PARAMS
from pathlib import Path
import os

REPO_CLONE_URL = 'https://github.com/aaandreyev/unet_seam.git'
REPO_CLONE_REF = 'main'
PROJECT_ROOT = Path('/content/unet_seam')

DRIVE_ROOT = Path('/content/drive/MyDrive/unet_seam_runs')
DRIVE_TRIPLETS_DIR = DRIVE_ROOT / 'synthetic_triplets_500'
DRIVE_TRIPLETS_ZIP = DRIVE_ROOT / 'synthetic_triplets_500.zip'
DRIVE_RUNS_PARENT = DRIVE_ROOT
DRIVE_SURGERY_ROOT = DRIVE_ROOT / 'model_surgery_runs'
# Optional: if you keep loose checkpoints outside run folders, point this to that directory.
# Leave as None when all checkpoints live under run_XXX/checkpoints.
DRIVE_LOCAL_CHECKPOINTS_DIR = None
RUN_NAME = 'surgery_run001'

USE_RAMDISK = True
RAMDISK_SIZE_GB = 8
COPY_RUNS_LOCALLY = True
COPY_CHECKPOINTS_LOCALLY = True
COPY_EXISTING_SURGERY_OUTPUTS = True
SYNC_INTERVAL_SEC = 180

USE_SHARDED_TRIPLETS = True
TRIPLET_SHARD_SIZE = 64
STAGES = 's0,s1,s2,s3,s4,s5'
TOP_K = 3
MAX_CYCLES = 4
TARGET_QUALITY_RELATIVE = 0.12
MINI_STRIPS = 20
FULL_STRIPS = 200
EVAL_BATCH_SIZE = None  # None -> auto by GPU type
EVAL_NUM_WORKERS = min(6, os.cpu_count() or 4)
MATERIALIZED_PRELOAD = True
ENABLE_FINETUNE_ON_PLATEAU = False
FINETUNE_PLATEAU_CYCLES = 3
FINETUNE_EPOCHS = 2
FINETUNE_LR = 5e-6

LOCAL_ROOT = Path('/content/ms_runtime')
LOCAL_DATA_ROOT = LOCAL_ROOT / 'data'
LOCAL_TRIPLETS_DIR = LOCAL_DATA_ROOT / 'synthetic_triplets_500'
LOCAL_TRIPLETS_SHARDED_DIR = LOCAL_DATA_ROOT / 'synthetic_triplets_500_sharded'
LOCAL_TRIPLETS_ZIP = LOCAL_DATA_ROOT / 'synthetic_triplets_500.zip'
LOCAL_OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'
LOCAL_RUNS_DIR = LOCAL_OUTPUTS_ROOT / 'unet_seam_runs'
LOCAL_CHECKPOINTS_DIR = LOCAL_OUTPUTS_ROOT / 'checkpoints'
LOCAL_SURGERY_OUTPUT_DIR = PROJECT_ROOT / 'model_surgery' / 'outputs_colab'
LOCAL_SURGERY_CFG = PROJECT_ROOT / 'runtime_configs' / 'model_surgery_colab.yaml'
DRIVE_RUN_DIR = DRIVE_SURGERY_ROOT / RUN_NAME
DRIVE_SURGERY_OUTPUT_DIR = DRIVE_RUN_DIR / 'model_surgery_outputs'


In [ ]:
# 1. MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
# 2. INSTALL / RUNTIME CHECKS
import json, platform, subprocess, sys
pkgs = ['pyyaml', 'tqdm', 'psutil', 'scikit-image', 'scipy', 'lpips', 'safetensors']
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'pigz'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(json.dumps({
    'python': sys.executable,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu_name': gpu_name,
}, ensure_ascii=False))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not found. In Colab select Runtime -> Change runtime type -> GPU and rerun.')
if EVAL_BATCH_SIZE is None:
    name = gpu_name.upper()
    if 'A100' in name:
        EVAL_BATCH_SIZE = 48
    elif 'L4' in name:
        EVAL_BATCH_SIZE = 24
    elif 'T4' in name:
        EVAL_BATCH_SIZE = 8
    else:
        EVAL_BATCH_SIZE = 8
print('EVAL_BATCH_SIZE =', EVAL_BATCH_SIZE)


In [ ]:
# 3. OPTIONAL RAMDISK
import os, psutil, shutil, subprocess
RAM_ROOT = Path('/content/ramdisk')
if USE_RAMDISK:
    mem = psutil.virtual_memory()
    avail_gb = int(mem.available / (1024 ** 3))
    eff = min(RAMDISK_SIZE_GB, max(4, avail_gb - 24))
    RAM_ROOT.mkdir(parents=True, exist_ok=True)
    mounted = subprocess.run(['mountpoint', str(RAM_ROOT)], capture_output=True).returncode == 0
    if not mounted:
        mount_result = subprocess.run(
            ['mount', '-t', 'tmpfs', '-o', f'size={eff}G,mode=777', 'tmpfs', str(RAM_ROOT)],
            text=True, capture_output=True,
        )
        if mount_result.returncode != 0:
            print('RAMDISK mount failed; fallback to /content:', mount_result.stderr[-500:])
            USE_RAMDISK = False
        else:
            print(f'Ramdisk mounted: {eff} GB at {RAM_ROOT}')
DATA_BASE = (RAM_ROOT / 'data') if USE_RAMDISK else LOCAL_DATA_ROOT
DATA_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_TRIPLETS_DIR = DATA_BASE / 'synthetic_triplets_500'
LOCAL_TRIPLETS_SHARDED_DIR = DATA_BASE / 'synthetic_triplets_500_sharded'
LOCAL_TRIPLETS_ZIP = DATA_BASE / 'synthetic_triplets_500.zip'
for p in [LOCAL_OUTPUTS_ROOT, LOCAL_RUNS_DIR, LOCAL_CHECKPOINTS_DIR, LOCAL_SURGERY_OUTPUT_DIR, LOCAL_SURGERY_CFG.parent]:
    p.mkdir(parents=True, exist_ok=True)
print('LOCAL_TRIPLETS_DIR =', LOCAL_TRIPLETS_DIR)


In [ ]:
# 4. PREFLIGHT VALIDATION
import json, os, shutil, zipfile
from pprint import pprint
import psutil

ALLOWED_STAGES = {'s0', 's1', 's2', 's3', 's4', 's5'}
REQUESTED_STAGES = [s.strip() for s in STAGES.split(',') if s.strip()]
UNKNOWN_STAGES = sorted(set(REQUESTED_STAGES) - ALLOWED_STAGES)
if UNKNOWN_STAGES:
    raise ValueError(f'Unknown stages: {UNKNOWN_STAGES}. Allowed: {sorted(ALLOWED_STAGES)}')
if not REQUESTED_STAGES:
    raise ValueError('STAGES is empty')
if not RUN_NAME.strip():
    raise ValueError('RUN_NAME must be non-empty')
if RUN_NAME in {'synthetic_triplets_500', 'model_surgery_runs'}:
    raise ValueError(f'RUN_NAME={RUN_NAME!r} conflicts with reserved names on Drive')
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f'DRIVE_ROOT not found: {DRIVE_ROOT}')
if not DRIVE_ROOT.is_dir():
    raise NotADirectoryError(DRIVE_ROOT)
triplet_source = None
triplet_mode = None
triplet_bytes = 0
if DRIVE_TRIPLETS_DIR.exists():
    if not DRIVE_TRIPLETS_DIR.is_dir():
        raise NotADirectoryError(DRIVE_TRIPLETS_DIR)
    if not (DRIVE_TRIPLETS_DIR / 'manifest.jsonl').exists():
        raise FileNotFoundError(DRIVE_TRIPLETS_DIR / 'manifest.jsonl')
    triplet_source = DRIVE_TRIPLETS_DIR
    triplet_mode = 'dir'
    triplet_bytes = sum(p.stat().st_size for p in DRIVE_TRIPLETS_DIR.rglob('*') if p.is_file())
elif DRIVE_TRIPLETS_ZIP.exists():
    if not DRIVE_TRIPLETS_ZIP.is_file():
        raise FileNotFoundError(DRIVE_TRIPLETS_ZIP)
    if not zipfile.is_zipfile(DRIVE_TRIPLETS_ZIP):
        raise RuntimeError(f'Invalid zip file: {DRIVE_TRIPLETS_ZIP}')
    with zipfile.ZipFile(DRIVE_TRIPLETS_ZIP, 'r') as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f'Corrupted zip entry in {DRIVE_TRIPLETS_ZIP}: {bad_member}')
        members = [m for m in zf.infolist() if not m.is_dir()]
        if not members:
            raise RuntimeError(f'Zip archive is empty: {DRIVE_TRIPLETS_ZIP}')
        names = {m.filename.rstrip('/') for m in members}
        has_manifest = 'manifest.jsonl' in names or 'synthetic_triplets_500/manifest.jsonl' in names
        if not has_manifest:
            raise RuntimeError(f'manifest.jsonl not found inside {DRIVE_TRIPLETS_ZIP}')
        triplet_source = DRIVE_TRIPLETS_ZIP
        triplet_mode = 'zip'
        zip_bytes = DRIVE_TRIPLETS_ZIP.stat().st_size
        unzip_bytes = sum(m.file_size for m in members)
        triplet_bytes = max(zip_bytes, unzip_bytes)
else:
    raise FileNotFoundError(f'Neither {DRIVE_TRIPLETS_DIR} nor {DRIVE_TRIPLETS_ZIP} exists')
run_dirs_on_drive = []
if DRIVE_RUNS_PARENT.exists():
    for run_dir in sorted(DRIVE_RUNS_PARENT.iterdir()):
        if not run_dir.is_dir():
            continue
        if run_dir.name in {'synthetic_triplets_500', 'model_surgery_runs'}:
            continue
        ckpt_dir = run_dir / 'checkpoints'
        ckpt_files = sorted(ckpt_dir.glob('*.pt')) if ckpt_dir.exists() else []
        if ckpt_files:
            run_dirs_on_drive.append(run_dir)
local_checkpoint_files = []
if DRIVE_LOCAL_CHECKPOINTS_DIR is not None:
    if not DRIVE_LOCAL_CHECKPOINTS_DIR.exists():
        raise FileNotFoundError(f'DRIVE_LOCAL_CHECKPOINTS_DIR not found: {DRIVE_LOCAL_CHECKPOINTS_DIR}')
    if not DRIVE_LOCAL_CHECKPOINTS_DIR.is_dir():
        raise NotADirectoryError(DRIVE_LOCAL_CHECKPOINTS_DIR)
    local_checkpoint_files = sorted(DRIVE_LOCAL_CHECKPOINTS_DIR.glob('*.pt'))
checkpoint_sources = len(run_dirs_on_drive) + (1 if local_checkpoint_files else 0)
if checkpoint_sources == 0:
    raise RuntimeError('No checkpoints found on Drive. Expected run_XXX/checkpoints/*.pt or DRIVE_LOCAL_CHECKPOINTS_DIR/*.pt')
target_usage_root = DATA_BASE if DATA_BASE.exists() else Path('/content')
free_bytes = shutil.disk_usage(target_usage_root).free
run_copy_bytes = 0
if COPY_RUNS_LOCALLY:
    run_copy_bytes = sum(p.stat().st_size for run_dir in run_dirs_on_drive for p in run_dir.rglob('*') if p.is_file())
checkpoint_copy_bytes = 0
if COPY_CHECKPOINTS_LOCALLY and DRIVE_LOCAL_CHECKPOINTS_DIR is not None:
    checkpoint_copy_bytes = sum(p.stat().st_size for p in DRIVE_LOCAL_CHECKPOINTS_DIR.rglob('*') if p.is_file())
estimated_need = triplet_bytes + run_copy_bytes + checkpoint_copy_bytes + 2 * 1024 ** 3
if estimated_need > free_bytes:
    raise RuntimeError(
        f'Not enough local space in {target_usage_root}: need ~{estimated_need / 1024**3:.1f} GB, '
        f'free ~{free_bytes / 1024**3:.1f} GB. Disable RAMDISK or reduce copied runs.'
    )
summary = {
    'drive_root': str(DRIVE_ROOT),
    'triplet_source': str(triplet_source),
    'triplet_mode': triplet_mode,
    'triplet_size_gb_est': round(triplet_bytes / 1024**3, 3),
    'run_dirs_found': [p.name for p in run_dirs_on_drive],
    'loose_checkpoints_found': len(local_checkpoint_files),
    'stages': REQUESTED_STAGES,
    'gpu_workers': EVAL_NUM_WORKERS,
    'batch_size': EVAL_BATCH_SIZE,
    'free_space_gb': round(free_bytes / 1024**3, 2),
    'ram_available_gb': round(psutil.virtual_memory().available / 1024**3, 2),
}
pprint(summary)


In [ ]:
# 5. CLONE REPOSITORY
import shutil, subprocess, sys
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
subprocess.run([
    'git', 'clone', '--depth', '1', '-b', REPO_CLONE_REF, REPO_CLONE_URL, str(PROJECT_ROOT)
], check=True)
if not (PROJECT_ROOT / 'model_surgery' / 'config.yaml').exists():
    raise FileNotFoundError(PROJECT_ROOT / 'model_surgery' / 'config.yaml')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# 6. PREPARE TRIPLETS + COPY RUNS / CHECKPOINTS / PREVIOUS OUTPUTS LOCALLY
import os, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def _copy_file(args):
    src, dst = args
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return src.stat().st_size

def copy_tree(src: Path, dst: Path, workers: int = 8, wipe: bool = True) -> None:
    if not src.exists():
        raise FileNotFoundError(src)
    if not src.is_dir():
        raise NotADirectoryError(src)
    if wipe and dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    files = [p for p in src.rglob('*') if p.is_file()]
    if not files:
        raise RuntimeError(f'No files found in {src}')
    total = sum(p.stat().st_size for p in files)
    with ThreadPoolExecutor(max_workers=max(1, min(workers, len(files) or 1))) as ex:
        it = ex.map(_copy_file, ((p, dst / p.relative_to(src)) for p in files), chunksize=8)
        copied = 0
        with tqdm(total=total, unit='B', unit_scale=True, desc=f'copy {src.name}') as pbar:
            for n in it:
                copied += n
                pbar.update(n)

def unzip_tree(zip_path: Path, dst: Path) -> None:
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f'Invalid zip file: {zip_path}')
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f'Corrupted zip entry in {zip_path}: {bad_member}')
        members = [m for m in zf.infolist() if not m.is_dir()]
        if not members:
            raise RuntimeError(f'Zip archive is empty: {zip_path}')
        root = dst.resolve()
        total = sum(m.file_size for m in members)
        with tqdm(total=total, unit='B', unit_scale=True, desc=f'unzip {zip_path.name}') as pbar:
            for member in members:
                out_path = (dst / member.filename).resolve()
                try:
                    out_path.relative_to(root)
                except ValueError as exc:
                    raise RuntimeError(f'Unsafe zip entry: {member.filename!r}') from exc
                out_path.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member, 'r') as src_f, out_path.open('wb') as dst_f:
                    while True:
                        chunk = src_f.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        dst_f.write(chunk)
                        pbar.update(len(chunk))

WORKERS = max(1, min(16, EVAL_NUM_WORKERS or 1, os.cpu_count() or 1))
if DRIVE_TRIPLETS_DIR.exists():
    copy_tree(DRIVE_TRIPLETS_DIR, LOCAL_TRIPLETS_DIR, workers=WORKERS, wipe=True)
elif DRIVE_TRIPLETS_ZIP.exists():
    shutil.copy2(DRIVE_TRIPLETS_ZIP, LOCAL_TRIPLETS_ZIP)
    unzip_tree(LOCAL_TRIPLETS_ZIP, LOCAL_TRIPLETS_DIR)
    nested = LOCAL_TRIPLETS_DIR / 'synthetic_triplets_500'
    if nested.exists() and (nested / 'manifest.jsonl').exists() and not (LOCAL_TRIPLETS_DIR / 'manifest.jsonl').exists():
        tmp_dst = LOCAL_TRIPLETS_DIR.with_name(LOCAL_TRIPLETS_DIR.name + '_tmp')
        if tmp_dst.exists():
            shutil.rmtree(tmp_dst)
        nested.rename(tmp_dst)
        shutil.rmtree(LOCAL_TRIPLETS_DIR)
        tmp_dst.rename(LOCAL_TRIPLETS_DIR)
else:
    raise FileNotFoundError(f'Neither {DRIVE_TRIPLETS_DIR} nor {DRIVE_TRIPLETS_ZIP} exists')
manifest_path = LOCAL_TRIPLETS_DIR / 'manifest.jsonl'
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)
manifest_lines = sum(1 for _ in manifest_path.open('r', encoding='utf-8'))
if manifest_lines == 0:
    raise RuntimeError(f'Empty manifest: {manifest_path}')
copied_run_names = []
if COPY_RUNS_LOCALLY and DRIVE_RUNS_PARENT.exists():
    for run_dir in sorted(DRIVE_RUNS_PARENT.iterdir()):
        if not run_dir.is_dir():
            continue
        if run_dir.name in {'synthetic_triplets_500', 'model_surgery_runs'}:
            continue
        ckpt_dir = run_dir / 'checkpoints'
        if not ckpt_dir.exists():
            continue
        if not list(ckpt_dir.glob('*.pt')):
            continue
        copy_tree(run_dir, LOCAL_RUNS_DIR / run_dir.name, workers=WORKERS, wipe=True)
        copied_run_names.append(run_dir.name)
if COPY_CHECKPOINTS_LOCALLY and DRIVE_LOCAL_CHECKPOINTS_DIR is not None and DRIVE_LOCAL_CHECKPOINTS_DIR.exists():
    copy_tree(DRIVE_LOCAL_CHECKPOINTS_DIR, LOCAL_CHECKPOINTS_DIR, workers=WORKERS, wipe=True)
if COPY_EXISTING_SURGERY_OUTPUTS and DRIVE_SURGERY_OUTPUT_DIR.exists():
    copy_tree(DRIVE_SURGERY_OUTPUT_DIR, LOCAL_SURGERY_OUTPUT_DIR, workers=WORKERS, wipe=False)
local_pt_files = list(LOCAL_RUNS_DIR.rglob('*.pt')) + list(LOCAL_CHECKPOINTS_DIR.rglob('*.pt'))
if not local_pt_files:
    raise RuntimeError('No local checkpoints after copy step. Notebook cannot run model_surgery.')
print('triplets:', LOCAL_TRIPLETS_DIR)
print('manifest lines:', manifest_lines)
print('runs:', LOCAL_RUNS_DIR)
print('copied run dirs:', copied_run_names)
print('local checkpoint files:', len(local_pt_files))
print('checkpoints:', LOCAL_CHECKPOINTS_DIR)
print('surgery outputs:', LOCAL_SURGERY_OUTPUT_DIR)


In [ ]:
# 7. OPTIONAL REPACK TO SHARDED TRIPLETS + BUILD RUNTIME CONFIG FOR MODEL SURGERY
import subprocess, sys, yaml
MATERIALIZED_DIR_FOR_SURGERY = LOCAL_TRIPLETS_DIR
cfg_path = PROJECT_ROOT / 'model_surgery' / 'config.yaml'
if not cfg_path.exists():
    raise FileNotFoundError(cfg_path)
manifest_path = LOCAL_TRIPLETS_DIR / 'manifest.jsonl'
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)
if USE_SHARDED_TRIPLETS:
    manifest_text = manifest_path.read_text(encoding='utf-8')
    if '"shard_path"' not in manifest_text[:4000] and 'shard_path' not in manifest_text[:4000]:
        subprocess.run([
            sys.executable, '-m', 'scripts.repack_materialized_triplets_to_shards',
            '--src', str(LOCAL_TRIPLETS_DIR),
            '--out', str(LOCAL_TRIPLETS_SHARDED_DIR),
            '--shard-size', str(TRIPLET_SHARD_SIZE),
            '--overwrite',
        ], cwd=str(PROJECT_ROOT), check=True)
        MATERIALIZED_DIR_FOR_SURGERY = LOCAL_TRIPLETS_SHARDED_DIR
    else:
        MATERIALIZED_DIR_FOR_SURGERY = LOCAL_TRIPLETS_DIR
if not (MATERIALIZED_DIR_FOR_SURGERY / 'manifest.jsonl').exists():
    raise FileNotFoundError(MATERIALIZED_DIR_FOR_SURGERY / 'manifest.jsonl')
print('MATERIALIZED_DIR_FOR_SURGERY =', MATERIALIZED_DIR_FOR_SURGERY)
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg['runs_dir'] = str(LOCAL_RUNS_DIR)
cfg['local_checkpoints_dir'] = str(LOCAL_CHECKPOINTS_DIR)
cfg['manifest'] = str(PROJECT_ROOT / 'manifests' / 'input_raw_manifest.jsonl')
cfg['output_dir'] = str(LOCAL_SURGERY_OUTPUT_DIR)
cfg['target_quality_relative'] = float(TARGET_QUALITY_RELATIVE)
cfg['eval']['materialized_dir'] = str(MATERIALIZED_DIR_FOR_SURGERY)
cfg['eval']['mini_strips'] = int(MINI_STRIPS)
cfg['eval']['full_strips'] = int(FULL_STRIPS)
cfg['eval']['batch_size'] = int(EVAL_BATCH_SIZE)
cfg['eval']['num_workers'] = int(EVAL_NUM_WORKERS)
cfg['eval']['materialized_preload'] = bool(MATERIALIZED_PRELOAD)
cfg['s5_cycle']['max_cycles'] = int(MAX_CYCLES)
cfg['s5_cycle']['top_k_base'] = int(TOP_K)
cfg['s5_cycle']['patience'] = min(int(cfg['s5_cycle'].get('patience', 3)), 3)
cfg['s5_cycle']['rerank_top_k'] = min(int(cfg['s5_cycle'].get('rerank_top_k', 4)), 4)
cfg['s5_cycle']['early_stop_no_improve_cycles'] = min(int(cfg['s5_cycle'].get('early_stop_no_improve_cycles', 2)), 2)
cfg['s5_cycle']['finetune_on_plateau']['enabled'] = bool(ENABLE_FINETUNE_ON_PLATEAU)
cfg['s5_cycle']['finetune_on_plateau']['plateau_cycles'] = int(FINETUNE_PLATEAU_CYCLES)
cfg['s5_cycle']['finetune_on_plateau']['epochs'] = int(FINETUNE_EPOCHS)
cfg['s5_cycle']['finetune_on_plateau']['lr'] = float(FINETUNE_LR)
cfg['s3_merge']['rerank_top_k'] = min(int(cfg['s3_merge'].get('rerank_top_k', 6)), 6)
cfg['s4_surgery']['rerank_top_k'] = min(int(cfg['s4_surgery'].get('rerank_top_k', 6)), 6)
enabled_stages = {s.strip() for s in STAGES.split(',') if s.strip()}
stage_map = {
    's0_survey': 's0',
    's1_gate_search': 's1',
    's2_ablation': 's2',
    's3_merge': 's3',
    's4_surgery': 's4',
    's5_cycle': 's5',
}
for key, short_name in stage_map.items():
    cfg['stages'][key] = short_name in enabled_stages
LOCAL_SURGERY_CFG.parent.mkdir(parents=True, exist_ok=True)
LOCAL_SURGERY_CFG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('runtime config ->', LOCAL_SURGERY_CFG)


In [ ]:
# 8. BACKGROUND SYNC TO DRIVE
import shutil, threading, time
if 'SYNC_STOP' in globals():
    SYNC_STOP.set()
SYNC_STOP = threading.Event()
DRIVE_SURGERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def sync_tree(src: Path, dst: Path) -> int:
    n = 0
    if not src.exists():
        return n
    for path in src.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(src)
        out = dst / rel
        tmp = out.with_name(f'.{out.name}.tmp')
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists() or out.stat().st_size != path.stat().st_size or out.stat().st_mtime < path.stat().st_mtime:
            shutil.copy2(path, tmp)
            tmp.replace(out)
            n += 1
    return n
def sync_all_to_drive():
    try:
        n = sync_tree(LOCAL_SURGERY_OUTPUT_DIR, DRIVE_SURGERY_OUTPUT_DIR)
        print('sync tick:', n, 'files ->', DRIVE_SURGERY_OUTPUT_DIR, flush=True)
    except Exception as exc:
        print('sync error:', repr(exc), flush=True)
def _sync_loop():
    while True:
        sync_all_to_drive()
        if SYNC_STOP.wait(SYNC_INTERVAL_SEC):
            break
sync_all_to_drive()
sync_thread = threading.Thread(target=_sync_loop, daemon=True)
sync_thread.start()
print('background sync started')


In [ ]:
# 9. RUN MODEL SURGERY
import os, subprocess, sys, threading
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(PROJECT_ROOT)
if not LOCAL_SURGERY_CFG.exists():
    raise FileNotFoundError(LOCAL_SURGERY_CFG)
if not (PROJECT_ROOT / 'model_surgery' / 'run.py').exists():
    raise FileNotFoundError(PROJECT_ROOT / 'model_surgery' / 'run.py')
if not (LOCAL_TRIPLETS_DIR / 'manifest.jsonl').exists() and not (LOCAL_TRIPLETS_SHARDED_DIR / 'manifest.jsonl').exists():
    raise RuntimeError('No prepared materialized dataset found locally before run step')
LOCAL_SURGERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
cmd = [
    sys.executable, '-u', '-m', 'model_surgery.run',
    '--config', str(LOCAL_SURGERY_CFG),
    '--stages', STAGES,
    '--top-k', str(TOP_K),
    '--max-cycles', str(MAX_CYCLES),
    '--output-dir', str(LOCAL_SURGERY_OUTPUT_DIR),
]
print('SURGERY CMD:', ' '.join(map(str, cmd)))
def _stream_cmd(cmd, cwd, env):
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def _pump():
        if p.stdout is not None:
            for line in p.stdout:
                print(line, end='', flush=True)
    t = threading.Thread(target=_pump, daemon=True)
    t.start()
    rc = p.wait()
    t.join(timeout=30)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd, output=f'model_surgery failed with exit code {rc}; inspect run.jsonl and notebook output above')
_stream_cmd(cmd, str(PROJECT_ROOT), env)


In [ ]:
# 10. FINAL SYNC + SUMMARY
if 'SYNC_STOP' in globals():
    SYNC_STOP.set()
if 'sync_all_to_drive' in globals():
    sync_all_to_drive()
print('Drive output dir =', DRIVE_SURGERY_OUTPUT_DIR)
print('Artifacts:')
if not DRIVE_SURGERY_OUTPUT_DIR.exists():
    raise FileNotFoundError(DRIVE_SURGERY_OUTPUT_DIR)
for path in sorted(DRIVE_SURGERY_OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(DRIVE_SURGERY_OUTPUT_DIR))
